# Persistance Interview Notes

Persistance is the core feature of LangGraph using which the state of the entire workflow is stored in a specific location.

Imp Pointer -> It can store the final as well as intermediate state values with the help of checkpointers.

4 Advantages of Persistance using checkpointers

1. Short Term Memory
2. Time Travel
3. HITL (Human in the loop)
4. Fault Tolerant (Most Important)

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI()

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the party? Because it knew it would be a "crust" favorite!',
 'explanation': 'This joke plays on words by using a pun. The word "crust" can refer to the outer layer of a pizza or a party favorite. By saying the pizza went to the party because it knew it would be a "crust" favorite, the joke is making a play on words to suggest that the pizza knew it would be a popular and well-liked food choice at the party. It\'s a light-hearted and playful way to incorporate food and humor into a joke.'}

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party? Because it knew it would be a "crust" favorite!', 'explanation': 'This joke plays on words by using a pun. The word "crust" can refer to the outer layer of a pizza or a party favorite. By saying the pizza went to the party because it knew it would be a "crust" favorite, the joke is making a play on words to suggest that the pizza knew it would be a popular and well-liked food choice at the party. It\'s a light-hearted and playful way to incorporate food and humor into a joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f198c95-b712-6cef-8002-4cdddc5257a8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-15T16:50:01.518786+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f198c95-abb7-65be-8001-4573867f91dc'}}, tasks=(), interrupts=())

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party? Because it knew it would be a "crust" favorite!', 'explanation': 'This joke plays on words by using a pun. The word "crust" can refer to the outer layer of a pizza or a party favorite. By saying the pizza went to the party because it knew it would be a "crust" favorite, the joke is making a play on words to suggest that the pizza knew it would be a popular and well-liked food choice at the party. It\'s a light-hearted and playful way to incorporate food and humor into a joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f198c95-b712-6cef-8002-4cdddc5257a8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-15T16:50:01.518786+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f198c95-abb7-65be-8001-4573867f91dc'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 

In [13]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti break up with the tortellini? \n\nBecause it was too saucy for them!',
 'explanation': 'This joke plays on the idea of "saucy" being a term used to describe someone who is too forward or flirtatious. In this case, the pasta sauce is being personified as being too saucy for the tortellini, causing them to break up. It\'s a light-hearted play on words that combines the literal meaning of pasta sauce with the figurative meaning of being overly assertive.'}

In [14]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the tortellini? \n\nBecause it was too saucy for them!', 'explanation': 'This joke plays on the idea of "saucy" being a term used to describe someone who is too forward or flirtatious. In this case, the pasta sauce is being personified as being too saucy for the tortellini, causing them to break up. It\'s a light-hearted play on words that combines the literal meaning of pasta sauce with the figurative meaning of being overly assertive.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f198c99-a101-6221-8002-9a742d924572'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-15T16:51:46.578868+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f198c99-92cd-6869-8001-d225bfbf818d'}}, tasks=(), interrupts=())

In [15]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the tortellini? \n\nBecause it was too saucy for them!', 'explanation': 'This joke plays on the idea of "saucy" being a term used to describe someone who is too forward or flirtatious. In this case, the pasta sauce is being personified as being too saucy for the tortellini, causing them to break up. It\'s a light-hearted play on words that combines the literal meaning of pasta sauce with the figurative meaning of being overly assertive.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f198c99-a101-6221-8002-9a742d924572'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-15T16:51:46.578868+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f198c99-92cd-6869-8001-d225bfbf818d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up